# Proof-of-Concept 4-Parameter Model K-Fold Cross-Validation

This notebook uses K-Fold Cross Validation on a small proof-of-concept emulator that predicts the **binned kSZ angular power spectrum ($D_\ell$)** using 4 reionization params ($z_{mean}$, $\alpha$, $k_b$, $b_0$).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

plt.rc("figure", figsize=(6, 4), dpi=150)

from reionemu import (
    DataLoaderConfig,
    FitConfig,
    FourParamEmulator,
    KFoldConfig,
    kfold_cross_validate,
    load_training_arrays,
)

## Condensed HDF5 Path
The condensed v6 simulation dataset is constructed with: $Y = \ln(D_\ell)$

In [ ]:
H5_PATH = Path("../data/processed/condensed_v6.h5").resolve()

## Define Model | Optimizer | Configs

In [ ]:
def model_builder() -> torch.nn.Module:
    return FourParamEmulator()


def optimizer_builder(model: torch.nn.Module) -> torch.optim.Optimizer:
    return torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)


loss_fn = torch.nn.MSELoss()

dlcfg = DataLoaderConfig(
    batch_size=32, seed=42, shuffle_train=True, normalize_X=True, normalize_Y=False
)

fitcfg = FitConfig(
    epochs=200, device="mps", early_stopping_patience=50, gradient_clipping=None
)

kcfg = KFoldConfig(k=5, seed=42, return_histories=True)

## Run K-Fold Cross Validation

In [ ]:
result = kfold_cross_validate(
    H5_PATH,
    model_builder=model_builder,
    optimizer_builder=optimizer_builder,
    loss_fn=loss_fn,
    kfold_config=kcfg,
    dl_config=dlcfg,
    fit_config=fitcfg,
)

## K-Fold CV Results

In [ ]:
print(result.keys())
print(f"Min Validation Loss Per Fold: {result['fold_best_val']}")
print(f"Mean Validation Loss (MSE): {result['mean_best_val']}")
print(f"STD Validation Loss: {result['std_best_val']}")

## Plot Learning Curves Per Fold

In [ ]:
histories = result["histories"]

plt.figure()
for i, h in enumerate(histories):
    plt.plot(h["val_loss"], label=f"fold {i + 1}")
plt.xlabel("Epoch")
plt.ylabel("Val Loss")
plt.title("Validation Loss Per Fold")
plt.legend()
plt.show()

## Best Validation Loss Per Fold

In [ ]:
best_vals = np.array(result["fold_best_val"], dtype=float)
plt.figure()
plt.bar(np.arange(1, len(best_vals) + 1), best_vals)
plt.xlabel("Fold")
plt.ylabel("Min Val Loss")
plt.title(
    f"Best Validation Loss Per Fold (mean={result['mean_best_val']:.4g}, std={result['std_best_val']:.4g})"
)
plt.show()

## Predict Function

In [ ]:
def predict(params, model, X_mean, X_std, Y_mean=None, Y_std=None, normalize_Y=True):
    params = (params - X_mean) / X_std

    xb = torch.from_numpy(params).to(device)

    model.eval()
    with torch.no_grad():
        pred_norm = model(xb).cpu().numpy()

    if normalize_Y:
        pred_log = pred_norm * Y_std + Y_mean
    else:
        pred_log = pred_norm

    pred_dl = np.exp(pred_log)
    return pred_dl

## Percent Errors of Folds

In [ ]:
X, Y, ell = load_training_arrays(H5_PATH)
device = torch.device("mps")

fold_pct_errors = []

for fold_idx in range(kcfg.k):
    fold_model = result["models"][fold_idx]
    fold_norm = result["norms"][fold_idx]
    val_indices = result["val_indices"][fold_idx]

    X_val = X[val_indices]
    Y_val = Y[val_indices]

    X_mean = fold_norm["X"].mean
    X_std = fold_norm["X"].std
    Y_mean = fold_norm["Y"].mean if dlcfg.normalize_Y else None
    Y_std = fold_norm["Y"].std if dlcfg.normalize_Y else None

    pred = predict(
        X_val, fold_model, X_mean, X_std, Y_mean, Y_std, normalize_Y=dlcfg.normalize_Y
    )
    true = np.exp(Y_val)

    pct_err = np.mean(np.abs((pred - true) / true)) * 100
    fold_pct_errors.append(pct_err)
    print(f"Fold {fold_idx + 1} % Error: {pct_err:.2f}%")

print()
print(f"CV Mean % Error: {np.mean(fold_pct_errors):.2f}%")
print(f"CV Std  % Error: {np.std(fold_pct_errors):.2f}%")